In [1]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import numpy as np
import re, math
import nltk
from nltk.tokenize import sent_tokenize

c:\Users\ranee\anaconda3\envs\amazonpacer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023",
                       "raw_review_All_Beauty",
                       trust_remote_code=True)

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True
)

c:\Users\ranee\anaconda3\envs\amazonpacer\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ranee\.cache\huggingface\hub\datasets--McAuley-Lab--Amazon-Reviews-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back

In [3]:
print("Reviews splits available:", reviews.keys())

Reviews splits available: dict_keys(['full'])


In [4]:
reviews["full"][1]

#asin : Amazon Standard Identification Number
# parent_asin : helps in joining reviews and meta data
#user_id : hashed amazon user id
#timestamp : review creation time
#helpful_vote : social signal / implicit relevance
#verified purchase : True if review is from a verified purchase

{'rating': 4.0,
 'title': 'Works great but smells a little weird.',
 'text': 'This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)',
 'images': [],
 'asin': 'B081TJ8YS3',
 'parent_asin': 'B081TJ8YS3',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': 1588615855070,
 'helpful_vote': 1,
 'verified_purchase': True}

In [5]:
meta[0]

#main_category : The high level amazon category the product belongs to
#average_rating : average rating of the product across all reviews
#rating_number : count of total ratings the product has received - helps in gauging popularity
#features : bullet points from the product listing
#description : product description text
#price : price of the product - string or numeric
# store: brand or seller of the product
# categories : hierarchical list of categories the product belongs to
# details : other product details like dimensions, weight, UPC: 12-digit barcode number used to uniquely identify a retail product in the U.S. and Canada. etc.
# parent_asin : for products that are variants (different size, color, etc.) of a main product, this field links to the main product's ASIN
# bought_together : list of ASINs that are frequently bought together with this product
# subtitles : list of available subtitles for the product listing

{'main_category': 'All Beauty',
 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)',
 'average_rating': 4.8,
 'rating_number': 10,
 'features': [],
 'description': [],
 'price': 'None',
 'images': {'hi_res': [None,
   'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'],
  'large': ['https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg',
   'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg'],
  'thumb': ['https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg',
   'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg'],
  'variant': ['MAIN', 'PT01']},
 'videos': {'title': [], 'url': [], 'user_id': []},
 'store': 'Howard Products',
 'categories': [],
 'details': '{"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}',
 'parent_asin': 'B01CUPMQZE',
 'bought_together': None,
 'subtitle': None,
 'author': None}

In [6]:
reviews.shape, meta.shape

({'full': (701528, 10)}, (112590, 16))

In [7]:
def hf_to_df(ds):
    # Turn an HF split into a pandas DataFrame, HF dataset behaves like a dict-of-columns
    return pd.DataFrame(ds)

def make_key(u, a, ts):
    # timestamps are strings in split files; standardize to str and then create a composite key string from user_id, parentasin, timestamp.
    return f"{u}::{a}::{str(ts)}"

reviews_full = reviews["full"]  # HF Dataset
keys = []
vals = []
# indexing raw reviews is just a himan readable label for the progress bar which is the description, so loop over every review row in reviews["full"]
for row in tqdm(reviews_full, desc="indexing raw reviews"):
    k = make_key(row.get("user_id"), row.get("parent_asin"), row.get("timestamp")) # build the composite key
    vals.append({
        "review_title": row.get("title"),
        "review_text": row.get("text"),
        "helpful_vote": row.get("helpful_vote"),
        "verified_purchase": row.get("verified_purchase"),
    }) # values that we care about from the reviews dataset
    keys.append(k)

review_lookup = pd.DataFrame(vals, index=keys)

review_lookup.head()

indexing raw reviews: 100%|██████████| 701528/701528 [00:38<00:00, 18238.03it/s]


,review_title,review_text,helpful_vote,verified_purchase
AGKHLEW2SOWHNMFQIJGBECAF7INQ::B00YQ6X8EO::1588687728923,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True
AGKHLEW2SOWHNMFQIJGBECAF7INQ::B081TJ8YS3::1588615855070,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True
AE74DYR3QUGVPZJ3P7RFWBGIX7XQ::B097R46CSY::1589665266052,Yes!,"Smells good, feels great!",2,True
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ::B09JS339BZ::1643393630220,Synthetic feeling,Felt synthetic,0,True
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ::B08BZ63GMJ::1609322563534,A+,Love it,0,True


In [8]:
meta_df = pd.DataFrame(meta)
meta_lookup = meta_df.set_index("parent_asin")[[
    "main_category","title","average_rating","rating_number","features", "description","price","store","categories","details","bought_together"
]].rename(columns={"title":"item_title"})

meta_lookup.head()

,main_category,item_title,average_rating,rating_number,features,description,price,store,categories,details,bought_together
parent_asin,,,,,,,,,,,
B01CUPMQZE,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],None,Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",None
B076WQZGPM,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],None,Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",None
B000B658RI,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],None,Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",None
B088FKY3VD,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],None,Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",None
B07NGFDN6G,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,None,Precision,[],"{""UPC"": ""644287689178""}",None


In [9]:
split_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"0core_timestamp_w_his_All_Beauty",
    trust_remote_code=True
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating train split: 583190 examples [00:03, 163059.33 examples/s]
Generating valid split: 71784 examples [00:01, 47917.72 examples/s]
Generating test split: 38955 examples [00:00, 49071.41 examples/s]


In [10]:
df = hf_to_df(split_ds["train"]).copy()

df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
if df["timestamp"].dropna().median() > 10**12:         # if looks like ms
    df["timestamp"] = (df["timestamp"] // 1000).astype("Int64") # divide 1000 to convert to seconds
else:
    df["timestamp"] = df["timestamp"].astype("Int64")


In [11]:
idx_series = review_lookup.index.to_series().astype(str)
keys = idx_series.str.split("::", n=2, expand=True)
keys.columns = ["user_id", "parent_asin", "timestamp"]


# Parse types- normalize revie timestamps too into SECONDS
keys["timestamp"] = pd.to_numeric(keys["timestamp"], errors="coerce")
# Align units to SECONDS like df
if keys["timestamp"].dropna().median() > 10**12:
    keys["timestamp"] = (keys["timestamp"] // 1000).astype("Int64")
else:
    keys["timestamp"] = keys["timestamp"].astype("Int64")

In [12]:
keys.reset_index(drop=True)

,user_id,parent_asin,timestamp
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,1588687728
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,1588615855
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,1589665266
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,1643393630
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,1609322563
...,...,...,...
701523,AFIXGFVEGLMOTMBTJL7H3VSIETDQ,B006YUIWKA,1478227021
701524,AFV7YZFOJF564EZGET5LG45K4QEA,B006YUIWKA,1480908730
701525,AHYDCWDMMVMLBX7FY7M7JKADKRDQ,B06ZZV9MZT,1590547974
701526,AF6ZIAEN7TQ2WY5ZL77F6JDPV7XQ,B000HB6VLE,1184798209


In [13]:
review_lookup

,review_title,review_text,helpful_vote,verified_purchase
AGKHLEW2SOWHNMFQIJGBECAF7INQ::B00YQ6X8EO::1588687728923,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True
AGKHLEW2SOWHNMFQIJGBECAF7INQ::B081TJ8YS3::1588615855070,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True
AE74DYR3QUGVPZJ3P7RFWBGIX7XQ::B097R46CSY::1589665266052,Yes!,"Smells good, feels great!",2,True
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ::B09JS339BZ::1643393630220,Synthetic feeling,Felt synthetic,0,True
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ::B08BZ63GMJ::1609322563534,A+,Love it,0,True
...,...,...,...,...
AFIXGFVEGLMOTMBTJL7H3VSIETDQ::B006YUIWKA::1478227021000,Four Stars,Conditioner is great shampoo not as I expected,0,True
AFV7YZFOJF564EZGET5LG45K4QEA::B006YUIWKA::1480908730000,Pretty,Did not work! Used the whole bottle and my hai...,0,False
AHYDCWDMMVMLBX7FY7M7JKADKRDQ::B06ZZV9MZT::1590547974067,Great sunless tanner,Product as expected. Shipping was on time.,0,True
AF6ZIAEN7TQ2WY5ZL77F6JDPV7XQ::B000HB6VLE::1184798209000,The Crown on top is a Ring!!!,"Not only is it a delicious fragrance, but also...",4,False


In [14]:
# Combine keys + values
reviews_text = pd.concat(
    [keys.reset_index(drop=True), review_lookup.reset_index(drop=True)],
    axis=1
)
reviews_text

,user_id,parent_asin,timestamp,review_title,review_text,helpful_vote,verified_purchase
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,1588687728,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,1588615855,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,1589665266,Yes!,"Smells good, feels great!",2,True
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,1643393630,Synthetic feeling,Felt synthetic,0,True
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,1609322563,A+,Love it,0,True
...,...,...,...,...,...,...,...
701523,AFIXGFVEGLMOTMBTJL7H3VSIETDQ,B006YUIWKA,1478227021,Four Stars,Conditioner is great shampoo not as I expected,0,True
701524,AFV7YZFOJF564EZGET5LG45K4QEA,B006YUIWKA,1480908730,Pretty,Did not work! Used the whole bottle and my hai...,0,False
701525,AHYDCWDMMVMLBX7FY7M7JKADKRDQ,B06ZZV9MZT,1590547974,Great sunless tanner,Product as expected. Shipping was on time.,0,True
701526,AF6ZIAEN7TQ2WY5ZL77F6JDPV7XQ,B000HB6VLE,1184798209,The Crown on top is a Ring!!!,"Not only is it a delicious fragrance, but also...",4,False


In [15]:
'''
Enrich a dataset split with review and metadata information.
'''
def enrich_split(split):
    """
    Join the split (HF dataset split) with:
      - review text fields from `review_lookup` (indexed by "user::parent::timestamp")
      - product meta from `meta_lookup` (indexed by parent_asin)
    WITHOUT changing any other helpers/globals.
    """
    import pandas as pd
    import numpy as np

    # --- 0) Convert splits to pandas dataframe
    df = hf_to_df(split).copy()

    # --- 1) Normalize split timestamp to SECONDS (handles ms vs s)
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    if df["timestamp"].dropna().median() > 10**12:         # if looks like ms
        df["timestamp"] = (df["timestamp"] // 1000).astype("Int64") # divide 1000 to convert to seconds
    else:
        df["timestamp"] = df["timestamp"].astype("Int64")

    # --- 2) Turn `review_lookup`'s composite index into 3 columns and dedupe
    # The index is "user_id::parent_asin::timestamp"
    idx_series = review_lookup.index.to_series().astype(str)
    keys = idx_series.str.split("::", n=2, expand=True)
    keys.columns = ["user_id", "parent_asin", "timestamp"]

    # Parse types- normalize revie timestamps too into SECONDS
    keys["timestamp"] = pd.to_numeric(keys["timestamp"], errors="coerce")
    # Align units to SECONDS like df
    if keys["timestamp"].dropna().median() > 10**12:
        keys["timestamp"] = (keys["timestamp"] // 1000).astype("Int64")
    else:
        keys["timestamp"] = keys["timestamp"].astype("Int64")

    # Combine keys + values
    reviews_text = pd.concat(
        [keys.reset_index(drop=True), review_lookup.reset_index(drop=True)],
        axis=1
    )

    # Dedupe on the 3-key join (keep the "best" first occurrence)
    # Heuristics if present: higher helpful_vote, longer text
    if "helpful_vote" in reviews_text.columns:
        reviews_text["_helpful_vote"] = pd.to_numeric(reviews_text["helpful_vote"], errors="coerce").fillna(0)
    else:
        reviews_text["_helpful_vote"] = 0
    if "review_text" in reviews_text.columns:
        reviews_text["_text_len"] = reviews_text["review_text"].fillna("").str.len()
    else:
        # if your columns are still 'title'/'text' (pre-rename), fallback
        txtcol = "review_text" if "review_text" in reviews_text.columns else ("text" if "text" in reviews_text.columns else None)
        reviews_text["_text_len"] = reviews_text.get(txtcol, pd.Series([""]*len(reviews_text))).fillna("").astype(str).str.len()

    reviews_text = reviews_text.sort_values(
        ["user_id","parent_asin","timestamp","_helpful_vote","_text_len"],
        ascending=[True, True, True, False, False]
    ).drop_duplicates(subset=["user_id","parent_asin","timestamp"], keep="first").drop(columns=["_helpful_vote","_text_len"])

    # --- 3) Left-join text fields by the 3-key
    df = df.merge(
        reviews_text,
        on=["user_id","parent_asin","timestamp"],
        how="left",
        validate="m:1"  # split keys should be unique; text side deduped to 1
    )

    # --- 4) Left-join META by parent_asin
    # meta_lookup is already indexed by parent_asin
    df = df.join(meta_lookup, on="parent_asin", how="left")

    # --- 5) Final tidying
    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    # ensure timestamp column stays integer-like
    df["timestamp"] = df["timestamp"].astype("Int64")

    return df

train_df = enrich_split(split_ds["train"])
valid_df = enrich_split(split_ds["valid"])
test_df = enrich_split(split_ds["test"])

In [16]:
train_df.columns

Index(['user_id', 'parent_asin', 'rating', 'timestamp', 'history',
       'review_title', 'review_text', 'helpful_vote', 'verified_purchase',
       'main_category', 'item_title', 'average_rating', 'rating_number',
       'features', 'description', 'price', 'store', 'categories', 'details',
       'bought_together'],
      dtype='object')

In [17]:
split_ds["train"]

Dataset({
    features: ['user_id', 'parent_asin', 'rating', 'timestamp', 'history'],
    num_rows: 583190
})

In [19]:
output_dir = ''

In [20]:

# Save train_df
train_df.to_csv(output_dir + 'train_df.csv', index=False)
print(f"train_df saved to {output_dir}train_df.csv")

# Save valid_df
valid_df.to_csv(output_dir + 'valid_df.csv', index=False)
print(f"valid_df saved to {output_dir}valid_df.csv")

# Save test_df
test_df.to_csv(output_dir + 'test_df.csv', index=False)
print(f"test_df saved to {output_dir}test_df.csv")

train_df saved to train_df.csv
valid_df saved to valid_df.csv
test_df saved to test_df.csv


In [21]:
train_df.head()

,user_id,parent_asin,rating,timestamp,history,review_title,review_text,helpful_vote,verified_purchase,main_category,item_title,average_rating,rating_number,features,description,price,store,categories,details,bought_together
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855,,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True,All Beauty,All Natural Vegan Dry Shampoo Powder - Eco Fri...,4.0,56,[],[],None,Two Goats Apothecary,[],"{""Brand"": ""Two Goats Apothecary"", ""Item Form"":...",None
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728,B081TJ8YS3,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True,All Beauty,Herbivore - Natural Sea Mist Texturizing Salt ...,4.3,384,[],"[If given the choice, weÕd leave most telltale...",None,HERBIVORE,[],"{""Hair Type"": ""Wavy"", ""Material Type Free"": ""D...",None
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266,,Yes!,"Smells good, feels great!",2,True,All Beauty,New Road Beauty - Creamsicle - Variety 3 Pack ...,4.4,699,"[Same Great Product, NEW PACKAGING., MOISTURIZ...",[New Road Beauty Paraffin Wax is recommended f...,21.98,New Road Beauty,[],"{""Package Dimensions"": ""10.5 x 6.4 x 1.6 inche...",None
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,1609322563,,A+,Love it,0,True,All Beauty,Yinhua Electric Nail Drill Kit Portable Profes...,3.5,20,[],[],None,Yinhua,[],"{""Package Dimensions"": ""8.5 x 3.82 x 2.24 inch...",None
4,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,1598567408,,Pretty Color,The polish was quiet thick and did not apply s...,0,True,All Beauty,"China Glaze Nail Polish, Wanderlust 1381",3.8,32,[Light lavender pink nail color with golden sh...,"[China Glaze Nail Polish, Wanderlust, 1381, .5...",7.1,China Glaze,[],"{""Brand"": ""China Glaze"", ""Item Form"": ""Liquid""...",None


In [22]:
train_df["review_text"][0]

'This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell like an orange coffee is offputting. (granted, I did know the smell was described but I was hoping it would be light)'

In [23]:
train_df["details"][1]

'{"Hair Type": "Wavy", "Material Type Free": "Dairy Free", "Scent": "Coconut", "Liquid Volume": "8 Fluid Ounces", "Item Form": "Spray", "Is Discontinued By Manufacturer": "No", "Package Dimensions": "9.57 x 3.27 x 3.15 inches; 15.84 Ounces", "UPC": "632709433010", "Manufacturer": "Herbivore Botanicals"}'

In [24]:
train_df.shape

(583190, 20)

In [25]:
train_df["user_id"].value_counts()

user_id
AG73BVBKUOH22USSFJA5ZWL7AKXA      156
AEZP6Z2C5AVQDZAJECQYZWQRNG3Q      132
AFDYIK3FNPY2JFBQYUWC6GSBMIRQ_2     83
AEMP3A7IKW37CMWFXNKXWW6HGJHA_1     73
AEOK4TQIKGO23SJKZ6PW4FETNNDA_1     67
                                 ... 
AG2EYGSZBHAOYPG2IPLXH3RAOWYA        1
AECZXE5ELSQIX4C2RKA2Z36HNHZA        1
AEHKDBA77OMG72SHVWOXXYRFPJZA        1
AHCXQQ2L7AHY6T5PDVVOGONZI6LA        1
AH4GCKLQC4SK7MUPUV3HX6MXI74Q        1
Name: count, Length: 534801, dtype: int64

In [26]:
train_df["parent_asin"].value_counts()

parent_asin
B085BB7B1M    1828
B0BM4GX6TT    1609
B07C533XCW    1310
B00R1TAN7I    1309
B019GBG0IE    1197
              ... 
B08KCY9N5K       1
B083FKK4PX       1
B08N5VRVMK       1
B094C7M695       1
B01ER73D5A       1
Name: count, Length: 95431, dtype: int64

In [27]:
#check if bought_together is not none anywhere
train_df["bought_together"].sum()

0

In [28]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ranee\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ranee\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

##### here we create sentence level dataset and then make a sentence level embedding matrix.

In [ ]:
# Building a sentence level dataset.
import pandas as pd
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import re

def clean_text(x: str) -> str:
    x = re.sub(r"\s+", " ", str(x)).strip() # removes leading and trailing whitespace characters from the resulting string: cleans text
    return x

def build_sentence_table(reviews_df: pd.DataFrame, max_sent_per_review=6, min_len=20):
    rows = [] # each element in rows is a dictionary
    for _, r in tqdm(reviews_df.iterrows(), total=len(reviews_df)): # here r= pandas series representing one review
        text = clean_text(r.get("review_text", "")) # helps clean review text, by editing whitespace, stripping leading and trailing spaces, converts non-strings safely
        if not text:
            continue
        sents = [s.strip() for s in sent_tokenize(text)] #NLTK splits review into sentences, ex: "I love the smell. But the bottle leaks." → ["I love the smell.", "But the bottle leaks."]
        # keep only reasonably informative sentences
        sents = [s for s in sents if len(s) >= min_len] # remove short sentences like okay. nice etc, short sentences cluster poorly.
        sents = sents[:max_sent_per_review] # some reviews are extremely long, without this, long reviews may dominate clustering

        for s in sents:
            rows.append({
                "parent_asin": r.get("parent_asin"),
                "user_id": r.get("user_id"),
                "rating": r.get("rating"),
                "verified_purchase": r.get("verified_purchase"),
                "sentence": s
            })
    return pd.DataFrame(rows)

sent_df = build_sentence_table(train_df, max_sent_per_review=10)
sent_df.head()

100%|██████████| 583190/583190 [01:56<00:00, 5022.97it/s]


,parent_asin,user_id,rating,verified_purchase,sentence
0,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,4.0,True,"This product does what I need it to do, I just..."
1,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,4.0,True,Having my head smell like an orange coffee is ...
2,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,4.0,True,"(granted, I did know the smell was described b..."
3,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,5.0,True,This spray is really nice.
4,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,5.0,True,"It smells really good, goes on really fine, an..."


In [ ]:
# LLMs cannot cluster, and clustering algorithms cannot understand language.

# Embed sentences: a library that wraps pretrained transformer models, specializes them for sentence-level semantic meaning, outputs fixed dimensional vectors.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2") # downloads a pretrained network, trained so that sentences with similar meaning, vectors close together.

sentences = sent_df["sentence"].tolist()

# for each sentence the model produces a vector E : sentence -> R^384, each row is a semantic representation of one sentence, normalize embedding so that every embedding on unit sphere
emb = embedder.encode(sentences, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10444 [00:00<?, ?it/s]

the above creates a very large embedding matix, lets try to reduce the size by creating review level embedding.

## 1. Embed the full review text directly (No sentence splitting)

In [30]:
!pip install sentence-transformers

In [31]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = train_df["review_text"].fillna("").astype(str).tolist()

# Optional: truncate super-long reviews (good for speed)
# MiniLM typically works best if you don't feed extremely long text
emb_reviews = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(emb_reviews.shape)  # (num_reviews, 384)

c:\Users\ranee\anaconda3\envs\amazonpacer\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ranee\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

(583190, 384)


In [32]:
output_dir = ''
np.save(output_dir + "review_embeddings.npy", emb_reviews)


In [33]:
texts = test_df["review_text"].fillna("").astype(str).tolist()

# Optional: truncate super-long reviews (good for speed)
# MiniLM typically works best if you don't feed extremely long text
emb_reviews = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(emb_reviews.shape)  # (num_reviews, 384)
np.save(output_dir + "review_embeddings_test.npy", emb_reviews)

Batches: 100%|██████████| 305/305 [02:29<00:00,  2.03it/s]


(38955, 384)


In [34]:
texts = valid_df["review_text"].fillna("").astype(str).tolist()

# Optional: truncate super-long reviews (good for speed)
# MiniLM typically works best if you don't feed extremely long text
emb_reviews = embedder.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(emb_reviews.shape)  # (num_reviews, 384)
np.save(output_dir + "review_embeddings_valid.npy", emb_reviews)

Batches: 100%|██████████| 561/561 [04:08<00:00,  2.26it/s]


(71784, 384)


In [ ]:
import numpy as np

# np.save(output_dir+"sentence_embeddings.npy", emb)
emb = np.load("sentence_embeddings.npy")
sent_df["embedding"] = emb.tolist()

In [ ]:
# cluster embeddings : using KMeans with K chosen by silhouette scoree

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def choose_k_by_silhouette(X, k_list):
    best_k, best_score = None, -1
    for k in k_list:
        km = KMeans(n_clusters=k, random_state=0, n_init="auto")
        lab = km.fit_predict(X)
        sc = silhouette_score(X, lab)
        if sc > best_score:
            best_k, best_score = k, sc
    return best_k, best_score

k, sc = choose_k_by_silhouette(emb, [6, 8, 10, 12, 16])
km = KMeans(n_clusters=k, random_state=0, n_init="auto")
sent_df["cluster_id"] = km.fit_predict(emb)

In [ ]:
# soft membership weights
import numpy as np

centers = km.cluster_centers_
# cosine distance since embeddings are normalized: dist = 1 - dot
sim = emb @ centers.T
dist = 1.0 - sim

tau = 20.0  # higher -> harder assignment
weights = np.exp(-tau * dist)
weights = weights / weights.sum(axis=1, keepdims=True)

sent_df["membership_max"] = weights.max(axis=1)


In [ ]:
from collections import defaultdict

def top_representatives(sent_df, emb, labels, top_m=12):
    reps = defaultdict(list)
    for cid in sorted(set(labels)):
        if cid == -1:
            continue
        idx = np.where(labels == cid)[0]
        if len(idx) == 0:
            continue
        # simple representativeness: average similarity to others (works for both)
        sub = emb[idx]
        sims = sub @ sub.T
        score = sims.mean(axis=1)
        top_idx = idx[np.argsort(-score)[:top_m]]
        reps[cid] = sent_df.iloc[top_idx]["sentence"].tolist()
    return reps

cluster_reps = top_representatives(sent_df, emb, sent_df["cluster_id"].values, top_m=10)
list(cluster_reps.items())[:1]

In [ ]:
!pip -q install transformers accelerate

from transformers import pipeline
gen = pipeline("text-generation", model="google/gemma-2-2b-it", device_map="auto")

def make_label_prompt(examples):
    bullets = "\n".join([f"- {s}" for s in examples[:10]])
    stringop =  f"""
                    You are labeling review-aspect clusters.
                    Given these review sentences, produce:
                    1) Aspect label: 3-5 words
                    2) 5 keywords
                    3) Polarity tendency: mostly_positive / mixed / mostly_negative
                    Return JSON with keys: label, keywords, polarity.

                    Sentences:
                    {bullets}
                  """.strip()
    return stringop

def label_with_local_llm(examples):
    prompt = make_label_prompt(examples)
    out = gen(prompt, max_new_tokens=120, do_sample=False)[0]["generated_text"]
    return out

# Example:
print(label_with_local_llm(cluster_reps[list(cluster_reps.keys())[0]]))


In [ ]:
aspect_map = {}
for cid, examples in cluster_reps.items():
    # replace with your chosen labeler
    # resp = call_llm(make_label_prompt(examples))
    # aspect_map[cid] = parse_json(resp)["label"]
    aspect_map[cid] = f"aspect_{cid}"  # placeholder

sent_df["aspect_label"] = sent_df["cluster_id"].map(aspect_map).fillna("noise/other")
sent_df[["cluster_id", "aspect_label", "sentence"]].head(20)
